# Course research · Kaggle T4 ×2

One interface for the existing course runner, feasibility gates, restartable experiments and offline results.
**Default: inspect only.** Each invocation performs one action. GPU calls, weight staging, freeze and main execution need their own explicit authorization below.

| Action | Accelerator | Internet | Result |
|---|---|---|---|
| `inspect` | None | Off | Verify this exact source package; show saved scope |
| `setup` | None | On | Install pinned candidate and stage the reviewed checkpoint; export reusable assets |
| `restore` | None | Off | Restore attached portable assets before assigning GPUs in a new session |
| `hardware` | T4 ×2 | Off | Native smoke, two 110,107-token resident-envelope diagnostics, bounded timeout, original evidence |
| `capability` | T4 ×2 | Off | First 90 development slots only |
| `pilot` | T4 ×2 | Off | Finish the same 1,114-generation / 1,178-use pilot |
| `freeze` | None | Off* | Audit pilot, review R7/timeout/family failures, forecast all 20 worlds; freeze if admitted |
| `main` | T4 ×2 | Off | Continue the fixed 5,752-generation / 6,280-use schedule for a bounded session |
| `status`, `analyze`, `export` | None | Off* | Monitor, independently audit/reproduce the report, or export without inference |

*A new CPU environment needs Internet for its first install; model weights are unnecessary for analysis.

Upload **course-kaggle-dataset.zip** as a private dataset, attach it and import this notebook.
In later sessions also attach **course-resume.zip**; GPU sessions reuse **course-assets.zip**.
Read `source/docs/KAGGLE_COURSE.md` for the short runbook and memory-review requirements.
The candidate requires Linux x86_64 Python 3.13 for GPU work. A different Kaggle image is a gate failure, not an automatic package upgrade.


In [1]:
# Edit this cell for the action you have authorized. Run All is safe with these defaults.
ACTION = "inspect"
ENABLE_GPU = False
PACKAGE_DIR = None  # Optional explicit directory containing source-manifest.json
RESUME_ZIP = None   # Otherwise finds the single attached course-resume.zip
ASSETS_ZIP = None   # Otherwise finds the single attached course-assets.zip


In [2]:
import hashlib,json,os,shutil,stat,sys,time,zipfile
from pathlib import Path,PurePosixPath
from IPython.display import display,FileLink,HTML
EXPECTED_MANIFEST_SHA256='c0f0b00f31b4090c7823d3dd6608980176d76b426663830a79c1fc4c401a6fe5'
EXPECTED_SOURCE_COMMIT='481699c6e46d5376ac451352d30b1a9da99ec0c3'
workspace=Path(os.environ.get('COURSE_WORKSPACE','/kaggle/working/course-workspace')).resolve()
workspace.mkdir(parents=True,exist_ok=True)
bundle=workspace/'bundle'
inputs=Path(os.environ.get('COURSE_INPUT','/kaggle/input'))
def attached(name,explicit=None):
    if explicit: return Path(explicit).resolve()
    matches=list(inputs.rglob(name))
    if len(matches)>1: raise ValueError('Select one exact '+name+' in the options cell')
    return matches[0] if matches else None
def extract_fresh(archive,target):
    if target.exists(): raise FileExistsError(target)
    with zipfile.ZipFile(archive) as z:
        names=set()
        for i in z.infolist():
            p=PurePosixPath(i.filename)
            if p.is_absolute() or '..' in p.parts or '\\' in i.filename or ':' in i.filename or i.filename in names or stat.S_ISLNK(i.external_attr>>16):
                raise ValueError('Unsafe/duplicated archive member')
            names.add(i.filename)
        target.mkdir(parents=True)
        z.extractall(target)
resume=attached('course-resume.zip',RESUME_ZIP)
if not bundle.exists():
    if resume: extract_fresh(resume,bundle)
    else: bundle.mkdir()
source=bundle/'source'
if not source.exists():
    options=[Path(PACKAGE_DIR)/'source-manifest.json'] if PACKAGE_DIR else list(inputs.rglob('source-manifest.json'))
    matches=[p for p in options if hashlib.sha256(p.read_bytes()).hexdigest()==EXPECTED_MANIFEST_SHA256]
    if len(matches)!=1: raise ValueError('Attach the exact course dataset ZIP (Kaggle extracts it), or set PACKAGE_DIR')
    package=matches[0].parent
    manifest=json.loads(matches[0].read_bytes())
    source.mkdir()
    for name,expected in manifest['members'].items():
        p=PurePosixPath(name)
        if p.is_absolute() or '..' in p.parts or '\\' in name or ':' in name: raise ValueError('Unsafe source member')
        original=package/name
        if not original.is_file() or original.is_symlink() or original.stat().st_size!=expected['bytes'] or hashlib.sha256(original.read_bytes()).hexdigest()!=expected['sha256']:
            raise ValueError('Source bytes differ: '+name)
        target=source/name
        target.parent.mkdir(parents=True,exist_ok=True)
        shutil.copyfile(original,target)
    shutil.copyfile(matches[0],source/'source-manifest.json')
manifest_bytes=(source/'source-manifest.json').read_bytes()
if hashlib.sha256(manifest_bytes).hexdigest()!=EXPECTED_MANIFEST_SHA256: raise ValueError('Resume requires the identical source package/notebook')
manifest=json.loads(manifest_bytes)
if (source/'SOURCE_COMMIT.txt').read_text().strip()!=EXPECTED_SOURCE_COMMIT: raise ValueError('Source commit differs')
for name,expected in manifest['members'].items():
    p=source/name
    if p.is_symlink() or not p.is_file() or p.stat().st_size!=expected['bytes'] or hashlib.sha256(p.read_bytes()).hexdigest()!=expected['sha256']:
        raise ValueError('Canonical source changed: '+name)
display(HTML('<div style="padding:16px;background:#e8f3f3;border-radius:10px"><b>Source verified</b> · '+EXPECTED_SOURCE_COMMIT[:12]+'<br>20 main worlds · 6,280 logical main uses · 5 separate development worlds</div>'))
display(FileLink(str(source/'docs/KAGGLE_COURSE.md')))


C:\Users\EndritHasani\Downloads\recovery_blueprint\recovery_blueprint\dist\course-kaggle-final-check\workspace\bundle\source\docs\KAGGLE_COURSE.md

## Configure the authorized action
Use `[lower, upper]` GPU-hour bounds supported by evidence. Zero also requires evidence. Preserve the same base bounds on every resume. Available account quota does not reset study spending.


In [3]:
# Restore prior bounds/reviews/authorizations; edit only current provider facts.
saved_config=bundle/'operator-config.json'
CONFIG=json.loads((saved_config if saved_config.exists() else source/'configs/course-kaggle.json').read_bytes())
CONFIG.update(execute_gpu=ENABLE_GPU,
    allocated_at=None,                  # Actual UTC provider assignment, e.g. YYYY-MM-DDTHH:MM:SS+00:00
    provider_quota_gpu_hours=None,       # Current provider allowance, independent of scientific spending
    provider_session_seconds=None,      # Current provider maximum session duration; verify it
    prior_sessions_released=CONFIG.get('prior_sessions_released',False),
    max_work_seconds=1800,
)
# Fill these overrides when first known. Empty dictionaries preserve saved values.
CONFIG['authorizations'].update({})  # keys: weights, hardware, pilot, freeze, main; explicit scope text
CONFIG['budget'].update({})         # all six prior_*_gpu_hours=[lower,upper], prior_smoke_attempts, prior_evidence
CONFIG['previous_session_release'].update({})  # released_at='...+00:00', evidence='actual provider deallocation receipt'
CONFIG['hardware_review'].update({})  # accepted, complete_live_memory_upper_bound_bytes, supported justification
CONFIG['pilot_review'].update({})     # family_condition_failures_reviewed, timeout_120_seconds_adequate
# Keep these same prior bounds on all resumes; hardware and pilot ledgers are added once by the runner.
config_path=bundle/'operator-config.json'
config_path.write_text(json.dumps(CONFIG,indent=2)+'\n')


1411

## Execute and monitor
The package streams progress every 15 seconds and exports quiescent SQLite backups at ten-minute boundaries. Interrupted/invalid replies remain recorded. A failed gate stops subsequent work.


In [4]:
import queue,threading,subprocess,signal
def bounded_child(arguments,limit):
    # The repo owns scientific work; this cell only streams its process output.
    log=workspace/('action-'+time.strftime('%Y%m%dT%H%M%S',time.gmtime())+'.log')
    process=subprocess.Popen([str(a) for a in arguments],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
        text=True,bufsize=1,start_new_session=(os.name!='nt'),cwd=workspace)
    lines=queue.Queue()
    def collect():
        for line in process.stdout: lines.put(line)
    reader=threading.Thread(target=collect,daemon=True); reader.start()
    deadline=time.monotonic()+limit
    try:
        with log.open('w',encoding='utf-8') as output:
            while process.poll() is None or not lines.empty():
                if time.monotonic()>=deadline: raise TimeoutError('Notebook controller deadline; preserve/export and Stop Session')
                try: line=lines.get(timeout=1)
                except queue.Empty: continue
                output.write(line); output.flush(); print(line,end='',flush=True)
    finally:
        if process.poll() is None:
            if os.name!='nt': os.killpg(process.pid,signal.SIGTERM)
            else: process.terminate()
            try: process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                if os.name!='nt': os.killpg(process.pid,signal.SIGKILL)
                else: process.kill()
                process.wait(timeout=5)
        reader.join(timeout=2)
    return process.returncode

if ACTION=='inspect':
    print('Verified upload package. GPU and weight staging remain disabled. Choose setup when resource evidence and weight staging authorization are ready.')
else:
    gpu_action=ACTION in ('hardware','capability','pilot','main')
    if gpu_action and not ENABLE_GPU: raise ValueError('Set ENABLE_GPU only for the separately authorized GPU action')
    if not gpu_action and Path('/proc/driver/nvidia/gpus').is_dir() and list(Path('/proc/driver/nvidia/gpus').iterdir()):
        raise ValueError('Use Accelerator None for CPU/setup/review/analysis actions; assigned GPU idle time remains chargeable')
    gpu_receipt=workspace/'gpu-environment'/'bootstrap'/'bootstrap.json'
    kind='setup' if ACTION=='setup' else 'gpu-environment' if ACTION=='restore' or gpu_action or gpu_receipt.exists() else 'cpu-environment'
    environment=workspace/kind
    receipt=environment/'bootstrap'/'bootstrap.json'
    if not receipt.exists():
        arguments=[sys.executable,'-I',source/'scripts/course_kaggle_bootstrap.py']
        if ACTION=='setup':
            arguments+=['setup','--config',config_path]
        elif ACTION=='restore':
            assets=attached('course-assets.zip',ASSETS_ZIP)
            if assets is None:
                options=[p.parent for p in inputs.rglob('prepared-assets.json') if (p.parent/'checkpoint').is_dir() and (p.parent/'wheels').is_dir()]
                if len(options)!=1: raise ValueError('Attach exactly one extracted course-assets dataset, or set ASSETS_ZIP to its directory/ZIP')
                assets=options[0]
            if assets is None: raise ValueError('Attach course-assets.zip from the CPU setup; use the same source package')
            arguments+=['restore','--assets',assets]
        elif gpu_action:
            raise ValueError('Run ACTION=restore with Accelerator None before assigning T4 x2, then change ACTION and record actual assignment time. This avoids spending GPU allowance on environment installation.')
        else:
            arguments+=['cpu']
        arguments+=['--root',source,'--scratch',environment/'scratch','--output',environment/'bootstrap']
        limit=2100
        if bounded_child(arguments,limit)!=0: raise RuntimeError('Environment setup/restore failed; evidence is retained in bootstrap.json and the action log')
    runtime=json.loads(receipt.read_bytes())
    if runtime.get('status')=='blocked': raise RuntimeError(runtime['reason'])
    if runtime.get('assets_receipt') and not runtime.get('requires_dataset_restore'):
        CONFIG.update(assets_receipt=runtime['assets_receipt'],hf_cache=runtime['hf_cache'])
        shutil.copyfile(runtime['assets_receipt'],bundle/'asset-receipt.json')
    elif (bundle/'asset-receipt.json').exists():
        CONFIG['assets_receipt']=str(bundle/'asset-receipt.json')
    config_path.write_text(json.dumps(CONFIG,indent=2)+'\n')
    if ACTION not in ('setup','restore'):
        arguments=[runtime['interpreter'],'-I',source/'scripts/course_kaggle_workflow.py',
            '--root',source,'--bundle',bundle,'--config',config_path,'--action',ACTION]
        limit=CONFIG['max_work_seconds']+180 if gpu_action else 2100
        code=bounded_child(arguments,limit)
        print('Action returned',code,'· inspect last-action.json for the exact gate/stop reason.')
    else:
        print('Pinned assets ready. Setup packs and consumes staging copies; download course-assets.zip and restore its private input in a new CPU session. Hardware acceptance remains open.')


Verified upload package. GPU and weight staging remain disabled. Choose setup when resource evidence and weight staging authorization are ready.


## Save evidence and release the session
Download the current ZIP and this executed notebook before stopping. Quick Save saves the notebook without a fresh execution; downloading exported evidence is the durable resume path. Save & Run All would start a new invocation.


In [5]:
# Run this cell after an action finishes/fails. It never invokes inference.
# Stdlib fallback also exports setup/refusal evidence before project installation.
import sqlite3,tempfile
from contextlib import closing
resume_export=workspace/'course-resume.zip'
partial=resume_export.with_suffix('.tmp')
with tempfile.TemporaryDirectory() as temporary:
    with zipfile.ZipFile(partial,'w',zipfile.ZIP_DEFLATED,compresslevel=3) as z:
        for p in sorted(bundle.rglob('*')):
            if not p.is_file() or p.is_symlink() or p.name.endswith(('-wal','-shm','.owner','.tmp','.pyc')) or '__pycache__' in p.parts: continue
            target=p
            if p.name in ('worker-0.sqlite','worker-1.sqlite'):
                target=Path(temporary)/(p.parent.name+'-'+p.name)
                with closing(sqlite3.connect(p.as_uri()+'?mode=ro',uri=True)) as original,closing(sqlite3.connect(target)) as copy:
                    original.backup(copy)
                    if copy.execute('PRAGMA integrity_check').fetchone()[0]!='ok': raise ValueError('SQLite backup failed')
            z.write(target,p.relative_to(bundle).as_posix())
partial.replace(resume_export)
for p in (workspace/'course-resume.zip',workspace/'setup'/'course-assets.zip'):
    if p.exists(): display(FileLink(str(p)))
if (bundle/'last-action.json').exists():
    display(FileLink(str(bundle/'last-action.json')))
if (bundle/'analysis-receipt.json').exists():
    analysis=json.loads((bundle/'analysis-receipt.json').read_bytes())
    display(FileLink(str(bundle/analysis['report'])))
    print(analysis['status'],analysis['coverage'])
print('Retain the ZIP(s) and actual executed notebook. For GPU actions: Stop Session now and verify provider deallocation. Record its UTC release and evidence for the next session.')


C:\Users\EndritHasani\Downloads\recovery_blueprint\recovery_blueprint\dist\course-kaggle-final-check\workspace\course-resume.zip

Retain the ZIP(s) and actual executed notebook. For GPU actions: Stop Session now and verify provider deallocation. Record its UTC release and evidence for the next session.
